# 03 · State change & ablation

`process(StateChange)` lesions a population of units, observes the effect, and restores exactly — the mechanism behind induced-dyslexia experiments. Here on a small torch network so it runs fast and is fully inspectable.

> Run on EC2 (GPU + Brain-Score data). Do **not** run on a laptop.


In [ ]:
import numpy as np, torch, torch.nn as nn
from brainscore_core.model_interface import (BrainScoreModel, StateChange,
    Selection, Perturbation)
from brainscore.perturbation import build_pytorch_ablation_fn
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))
net.eval()

## Wrap with a state_change_fn and ablate a population
Layer `'0'` is the first Linear (16 output units); we zero its first 8.

In [ ]:
bs = BrainScoreModel(identifier='toy', model=net, region_layer_map={'R': '0'},
                     preprocessors={}, activations_model=None,
                     state_change_fn=build_pytorch_ablation_fn(net))
x = torch.randn(5, 8)
baseline = net(x).detach().numpy()
applied = bs.process(StateChange(kind='ablation',
             target=Selection(layer='0', indices=list(range(8))),
             perturbation=Perturbation(kind='zero')))
lesioned = net(x).detach().numpy()
print('handle:', applied.handle_id,
      '| output changed:', not np.allclose(baseline, lesioned))

## Visualize intact vs lesioned vs difference

In [ ]:
from brainscore.visualization import before_after_difference
# show first-layer activations under a hook is overkill here; show outputs
before_after_difference(baseline, lesioned, out_png='ablation.png',
                        title='Output before/after lesion')
from IPython.display import Image; Image('ablation.png')

## Restore exactly and confirm bit-for-bit recovery

In [ ]:
bs.reset()
restored = net(x).detach().numpy()
print('restored == baseline:', np.allclose(restored, baseline))

**Null for ablation:** ablating a *random* same-size population should produce a weaker effect than a curated (e.g. top-Cohen's-d) population. Use `brainscore_core.nulls.random_unit_subset` to pick the random set.